# Projet final : les récoltes agricoles

**Équipe :** (nom de l'équipe et des 5 membres)

Une direction départementale de l'agriculture (fictive) vous confie trois ans de récoltes, de 2023 à 2025, par saison (printemps, automne), département et culture, avec la pluie tombée pendant chaque saison. Le ministère veut savoir quelles cultures pèsent le plus, où les rendements sont les meilleurs, et si la pluie explique les mauvaises années.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/recoltes_brut.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(376, 6)`, sans valeur manquante ni doublon, avec 2 valeurs dans `saison`, 10 valeurs dans `departement`, 7 valeurs dans `culture`.

## 3. Jointures

Joignez `propre` avec `cultures.csv`, `pluies.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 376 lignes après les jointures, autant qu'avant.

In [ ]:
# Partie 2 — Jointures et indicateurs réalisée par REJUSTE Jovens Fequiere.

cultures = pd.read_csv("donnees/cultures.csv")
pluies = pd.read_csv("donnees/pluies.csv")
departements = pd.read_csv("donnees/departements.csv")

# Chaque table de référence doit avoir une seule ligne par clé de jointure.
assert not cultures["culture"].duplicated().any()
assert not pluies.duplicated(["annee", "saison", "departement"]).any()
assert not departements["departement"].duplicated().any()

lignes_avant = len(propre)
complet = propre.merge(
    cultures,
    on="culture",
    how="left",
    validate="many_to_one",
    indicator="jointure_culture",
)
print(f"Jointure cultures : {lignes_avant} lignes avant, {len(complet)} après")
if not complet["jointure_culture"].eq("both").all():
    absentes = complet.loc[complet["jointure_culture"] != "both", "culture"].unique()
    raise ValueError(f"Cultures sans correspondance : {absentes.tolist()}")
complet = complet.drop(columns="jointure_culture")
assert len(complet) == lignes_avant

lignes_avant = len(complet)
complet = complet.merge(
    pluies,
    on=["annee", "saison", "departement"],
    how="left",
    validate="many_to_one",
    indicator="jointure_pluie",
)
print(f"Jointure pluies : {lignes_avant} lignes avant, {len(complet)} après")
if not complet["jointure_pluie"].eq("both").all():
    absentes = complet.loc[
        complet["jointure_pluie"] != "both",
        ["annee", "saison", "departement"],
    ].drop_duplicates()
    raise ValueError(f"Récoltes sans mesure de pluie :\n{absentes}")
complet = complet.drop(columns="jointure_pluie")
assert len(complet) == lignes_avant

lignes_avant = len(complet)
complet = complet.merge(
    departements,
    on="departement",
    how="left",
    validate="many_to_one",
    indicator="jointure_departement",
)
print(f"Jointure départements : {lignes_avant} lignes avant, {len(complet)} après")
if not complet["jointure_departement"].eq("both").all():
    absents = complet.loc[
        complet["jointure_departement"] != "both", "departement"
    ].unique()
    raise ValueError(f"Départements sans correspondance : {absents.tolist()}")
complet = complet.drop(columns="jointure_departement")
assert len(complet) == lignes_avant == 376

# Ces indicateurs sont réutilisés par les analyses et le tableau de bord.
complet["rendement_t_ha"] = complet["production_t"] / complet["surface_ha"]
complet["valeur_htg"] = (
    complet["production_t"] * 1000 * complet["prix_htg_kg"]
)
complet["kg_par_habitant"] = (
    complet["production_t"] * 1000 / complet["population"]
)

print(
    "Rendement moyen pondéré : "
    f"{complet['production_t'].sum() / complet['surface_ha'].sum():.2f} t/ha"
)
print(f"Valeur totale estimée : {complet['valeur_htg'].sum():,.0f} HTG")
complet.head()

Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/recoltes_propre.csv", index=False)
```

In [ ]:
complet.to_csv("donnees/recoltes_propre.csv", index=False)
print(
    "Fichier donnees/recoltes_propre.csv enregistré : "
    f"{len(complet)} lignes et {len(complet.columns)} colonnes."
)

## 4. Quelles cultures pèsent le plus, en tonnes puis en valeur (gourdes) ? Le classement change-t-il ?

**Ce qu'on en conclut :**

## 5. Où les rendements (tonnes par hectare) sont-ils les meilleurs, culture par culture ?

**Ce qu'on en conclut :**

## 6. La pluie explique-t-elle les rendements ? Pour quelles cultures ? Montrez-le avec des nuages de points.

**Ce qu'on en conclut :**

## 7. Quelles cultures progressent ou reculent entre 2023 et 2025 ?

**Ce qu'on en conclut :**

## 8. Rapportez la production à la population de chaque département : quels départements produisent le plus par habitant ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.